# 08 · Extract tables and charts

**Question.** Can response-masked training preserve line items, totals and chart values on new layouts?

Keep provenance, extraction fidelity and arithmetic agreement separate. A printed total can be inconsistent and still be read faithfully.

[Open the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/08-extract-tables-and-charts).

## Which software and hardware does this lab use?

The same quantised base and processor serve every branch. Only language-backbone adapters train. The notebook stops early if CUDA is unavailable.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0",
                "bitsandbytes==0.50.2", "pillow==11.3.0", "datasets==3.6.0",
                "jiwer==4.0.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0', 'bitsandbytes==0.50.2', 'pillow==11.3.0', 'datasets==3.6.0', 'jiwer==4.0.0'], returncode=0)

## How are invoice families held out?

Original synthetic records are split by supplier-and-layout family before rendering. The same example retains its family identifier across image derivatives.

In [2]:
"""Versioned synthetic invoices and independent record scoring for the course."""

from __future__ import annotations

import json
import random
import re
from collections import Counter
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Any

CATALOG_VERSION = "course-invoice-v1"
SPLIT_SIZES = {"train": 576, "dev": 72, "familiar": 96, "novel": 96, "independent": 48, "release": 120}
VISIBLE_CONTRACT = (
    "Extract exactly one JSON object with invoice_id, issued_on (YYYY-MM-DD), supplier, "
    "currency (ISO code), line_items (description, quantity, unit_minor), subtotal_minor, "
    "tax_minor, and total_minor. Preserve line order and convert currency amounts to integer cents. "
    "Use null only for a missing or ambiguous field. Do not rewrite printed amounts, invent fields, or add explanations."
)

@dataclass(frozen=True)
class Line:
    description: str
    quantity: int
    unit_minor: int


@dataclass(frozen=True)
class Invoice:
    key: str
    family: str
    supplier: str
    issued_on: str
    currency: str
    items: tuple[Line, ...]
    tax_minor: int | None
    locale: str
    split: str
    layout: str = "register"
    printed_total_minor: int | None = None


def invoice_record(invoice: Invoice) -> dict[str, Any]:
    subtotal = sum(line.quantity * line.unit_minor for line in invoice.items)
    total = subtotal + invoice.tax_minor if invoice.tax_minor is not None else None
    if invoice.printed_total_minor is not None:
        total = invoice.printed_total_minor
    return {
        "invoice_id": invoice.key,
        "issued_on": invoice.issued_on,
        "supplier": invoice.supplier,
        "currency": invoice.currency,
        "line_items": [
            {"description": item.description, "quantity": item.quantity, "unit_minor": item.unit_minor}
            for item in invoice.items
        ],
        "subtotal_minor": subtotal,
        "tax_minor": invoice.tax_minor,
        "total_minor": total,
    }


def make_catalog() -> list[Invoice]:
    rng = random.Random(20261126)
    sources = {
        "train": (24, ("register", "memo", "checkout", "ledger")),
        "dev": (6, ("register", "memo", "checkout", "ledger")),
        "familiar": (8, ("register", "memo", "checkout", "ledger")),
        "novel": (8, ("packing", "remittance", "service", "notice")),
        "independent": (4, ("separate",)),
        "release": (10, ("packing", "remittance", "service", "notice", "separate")),
    }
    locales = (("en-US", "USD"), ("en-GB", "GBP"), ("de-DE", "EUR"), ("fr-FR", "EUR"), ("en-US", "CAD"))
    services = ("Cooling filter", "Cable sleeve", "Bench calibration", "Door seal", "Grounding clip",
                "Rack inspection", "Meter label", "Packing insert", "Carrier adjustment", "Mounting plate")
    catalog = []
    seen_keys = set()
    supplier_codes = rng.sample(range(1000, 9999), sum(n for n, _ in sources.values()))
    supplier_index = 0
    for split, (family_count, layouts) in sources.items():
        for family_number in range(family_count):
            supplier_code = supplier_codes[supplier_index]
            supplier_index += 1
            layout = layouts[family_number % len(layouts)]
            family = f"{split}-{family_number:02d}"
            locale, currency = locales[family_number % len(locales)]
            for row in range(SPLIT_SIZES[split] // family_count):
                code = f"SF-{rng.randrange(100000, 1000000)}"
                while code in seen_keys:
                    code = f"SF-{rng.randrange(100000, 1000000)}"
                seen_keys.add(code)
                items = tuple(
                    Line(kind, rng.randint(1, 9), rng.randrange(200, 24800))
                    for kind in rng.sample(services, rng.randrange(1, 4))
                )
                subtotal = sum(x.quantity * x.unit_minor for x in items)
                tax = (subtotal * (7 if currency == "USD" else 13) + 50) // 100
                if split in {"novel", "independent", "release"} and row % 12 == 4:
                    tax = None
                displayed = subtotal + tax + 101 if tax is not None and row % 12 == 7 else None
                issued_on = (date(2025, 1, 1) + timedelta(days=rng.randint(0, 590))).isoformat()
                catalog.append(Invoice(code, family, f"Sample Supplier {supplier_code}", issued_on,
                                       currency, items, tax, locale, split, layout, displayed))
    return catalog


def formatted_money(minor: int | None, locale: str) -> str:
    if minor is None:
        return "-"
    dollars, cents = divmod(minor, 100)
    number = f"{dollars:,}.{cents:02d}"
    if locale == "de-DE":
        return number.replace(",", "_").replace(".", ",").replace("_", ".")
    if locale == "fr-FR":
        return number.replace(",", " ").replace(".", ",")
    return number


def invoice_text(invoice: Invoice) -> str:
    gold = invoice_record(invoice)
    money = lambda value: formatted_money(value, invoice.locale)
    identifier = f"{invoice.key}"
    supplier = invoice.supplier
    issued = invoice.issued_on
    currency = invoice.currency
    subtotal = money(gold["subtotal_minor"])
    tax = money(gold["tax_minor"])
    total = money(gold["total_minor"])
    register = [f"{row.description} | {row.quantity} | {money(row.unit_minor)}" for row in invoice.items]
    inline = [f"{row.quantity} x {row.description} @ {money(row.unit_minor)} per unit" for row in invoice.items]
    service = [f"{row.description}: {money(row.unit_minor)} each for {row.quantity} units" for row in invoice.items]
    if invoice.layout == "register":
        return "\n".join([f"Supplier: {supplier}", f"Invoice: {identifier}", f"Issued: {issued}", f"Currency: {currency}",
                          "Description | Quantity | Unit price", *register,
                          f"Subtotal: {subtotal}", f"Tax: {tax}", f"Amount due: {total}"])
    if invoice.layout == "memo":
        return "\n".join([f"{supplier} billing memo / {issued}", f"Account invoice {identifier} [{currency}]",
                          *inline, f"Total due {total} including tax {tax}; net amount {subtotal}."])
    if invoice.layout == "checkout":
        return "\n".join([f"{identifier} / supplier {supplier}", f"Date of issue {issued} / All amounts {currency}",
                          *service, f"Net {subtotal}", f"Sales tax {tax}", f"Payable {total}"])
    if invoice.layout == "ledger":
        return "\n".join([f"{supplier} | issued {issued} | invoice {identifier}", f"Ledger in {currency}",
                          "Service | Count | Per unit", *register, f"Before tax {subtotal}",
                          f"Tax charged {tax}", f"Balance {total}"])
    if invoice.layout == "packing":
        return "\n".join([f"Billing details / {identifier}", f"Sold by {supplier}", f"Document date {issued}",
                          f"Invoice currency {currency}", f"Balance payable {total}", *service,
                          f"Merchandise {subtotal}", f"Tax {tax}"])
    if invoice.layout == "remittance":
        return "\n".join([f"Remittance advice for {identifier} ({currency})", f"{supplier} / Document issued {issued}",
                          f"Pay this amount {total}", *inline, f"Net charges {subtotal} / Tax {tax}"])
    if invoice.layout == "service":
        return "\n".join([f"Ref {identifier} / {currency}", f"Service fee notice, {issued}", supplier,
                          *[f"Unit {money(row.unit_minor)} / {row.quantity} times / {row.description}" for row in invoice.items],
                          f"Fee subtotal {subtotal}", f"Plus tax {tax}", f"Amount requested {total}"])
    if invoice.layout == "notice":
        return "\n".join([f"{supplier} / statement {identifier}", f"Issued on {issued}",
                          f"Invoice denomination {currency}", "One-off charges", *service,
                          f"Pre-tax {subtotal} / Tax assessment {tax} / Final bill {total}"])
    return "\n".join([f"Document {identifier}", f"Biller {supplier} / As of {issued} / {currency}", *inline,
                      f"Gross {total}", f"Breakdown: net {subtotal} and taxes {tax}."])


def _strict_dict(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key {key!r}")
        result[key] = value
    return result


def parse_record(completion: str | dict[str, Any]) -> dict[str, Any] | None:
    if isinstance(completion, str):
        try:
            data = json.loads(completion, object_pairs_hook=_strict_dict)
        except (ValueError, TypeError, RecursionError):
            return None
    else:
        data = completion
    keys = {"invoice_id", "issued_on", "supplier", "currency", "line_items", "subtotal_minor", "tax_minor", "total_minor"}
    if not isinstance(data, dict) or set(data) != keys:
        return None
    for name in ("invoice_id", "supplier", "issued_on", "currency"):
        value = data[name]
        if value is not None and (not isinstance(value, str) or not value.strip()):
            return None
    try:
        if data["issued_on"] is not None and date.fromisoformat(data["issued_on"]).isoformat() != data["issued_on"]:
            return None
    except ValueError:
        return None
    if data["currency"] is not None and re.fullmatch("[A-Z]{3}", data["currency"]) is None:
        return None
    for name in ("subtotal_minor", "tax_minor", "total_minor"):
        if data[name] is not None and type(data[name]) is not int:
            return None
    if not isinstance(data["line_items"], list):
        return None
    for row in data["line_items"]:
        if not isinstance(row, dict) or set(row) != {"description", "quantity", "unit_minor"}:
            return None
        if type(row["description"]) is not str or type(row["quantity"]) is not int or type(row["unit_minor"]) is not int:
            return None
        if not row["description"].strip() or row["quantity"] <= 0:
            return None
    return data


def score_record(completion: str | dict[str, Any], invoice: Invoice) -> dict[str, Any]:
    truth = invoice_record(invoice)
    observed = parse_record(completion)
    field_correct = {key: observed is not None and observed.get(key) == value
                     for key, value in truth.items() if key != "line_items"}
    keys = ("description", "quantity", "unit_minor")
    expected = Counter(tuple(r[k] for k in keys) for r in truth["line_items"])
    predicted = Counter(tuple(r[k] for k in keys) for r in observed["line_items"]) if observed is not None else Counter()
    matches = (expected & predicted).total()
    row_count = expected.total() + predicted.total()
    return {"valid": observed is not None, "exact": observed == truth,
            "fields": sum(field_correct.values()) / len(field_correct), "field_correct": field_correct,
            "row_f1": 2 * matches / row_count if row_count else 1.0}


def rules_extract(text: str) -> dict[str, Any] | None:
    lines = text.splitlines()
    if len(lines) < 9 or lines[4] != "Description | Quantity | Unit price":
        return None
    property_names = ("supplier", "invoice_id", "issued_on", "currency")
    headers = ("Supplier: ", "Invoice: ", "Issued: ", "Currency: ")
    if any(not line.startswith(header) for line, header in zip(lines, headers)):
        return None
    data = {name: line.removeprefix(header) for name, line, header in zip(property_names, lines, headers)}

    def minor(text_amount: str) -> int | None:
        if text_amount == "-":
            return None
        if re.fullmatch(r"(?:\d+|\d{1,3}(?:[., ]\d{3})+)[.,]\d{2}", text_amount) is None:
            raise ValueError("Unsupported amount format")
        units, frac = text_amount[:-3], text_amount[-2:]
        return int(re.sub(r"\D", "", units)) * 100 + int(frac)

    try:
        rows = [line.split(" | ") for line in lines[5:-3]]
        data["line_items"] = [{"description": desc, "quantity": int(qty), "unit_minor": minor(unit)}
                              for desc, qty, unit in rows]
        data["subtotal_minor"] = minor(lines[-3].removeprefix("Subtotal: "))
        data["tax_minor"] = minor(lines[-2].removeprefix("Tax: "))
        data["total_minor"] = minor(lines[-1].removeprefix("Amount due: "))
    except ValueError:
        return None
    return parse_record(data)


def examples_for(split: str, limit: int | None = None) -> list[Invoice]:
    found = [invoice for invoice in make_catalog() if invoice.split == split]
    if not found:
        raise ValueError(f"Unknown invoice split {split!r}")
    if limit is None or limit >= len(found):
        return found
    if limit <= 0:
        raise ValueError("Limit must be positive")
    groups: dict[str, list[Invoice]] = {}
    for invoice in found:
        groups.setdefault(invoice.family, []).append(invoice)
    balanced = [invoice for group_rows in zip(*groups.values(), strict=True) for invoice in group_rows]
    return balanced[:limit]

In [3]:
"""Original invoice page and chart rendering for the document notebooks."""

from PIL import Image, ImageDraw, ImageFont

# The notebooks inline this file after the fixture contract. The fallback import
# keeps the same functions usable in ordinary Python.
if "invoice_text" not in globals():
    from invoice_fixtures import Invoice, invoice_record, invoice_text


def _font(size: int, bold: bool = False):
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    try:
        return ImageFont.truetype(name, size)
    except OSError:
        return ImageFont.load_default(size=size)


def invoice_image(invoice: Invoice) -> Image.Image:
    image = Image.new("RGB", (896, 624), "#faf9f5")
    draw = ImageDraw.Draw(image)
    fields = invoice_text(invoice).splitlines()
    accents = {
        "register": "#245067", "memo": "#732f41", "checkout": "#30614e", "ledger": "#2a3e72",
        "packing": "#31535c", "remittance": "#694323", "service": "#4b3f65", "notice": "#73422d",
    }
    accent = accents.get(invoice.layout, "#4a5744")
    draw.rounded_rectangle((32, 29, 864, 595), 11, fill="white", outline="#d9dad8", width=2)
    draw.rectangle((33, 34, 863, 42), fill=accent)
    draw.text((64, 63), "Supplier billing", font=_font(28, bold=True), fill=accent)
    draw.text((64, 102), "Original synthetic course fixture", font=_font(15), fill="#5b6462")
    draw.line((61, 134, 835, 134), fill="#d5dad8", width=2)
    for i, line in enumerate(fields):
        y = 153 + 35 * i
        if i % 2:
            draw.rectangle((58, y - 3, 837, y + 28), fill="#f5f7f6")
        is_heading = line.startswith(("Description |", "Service |", "One-off charges"))
        draw.text((65, y), line, fill="#233034", font=_font(20, bold=is_heading))
        if draw.textbbox((65, y), line, font=_font(20, bold=is_heading))[2] >= 837:
            raise ValueError(f"Invoice field did not fit the page: {line!r}")
    return image


def spend_chart(invoice: Invoice, *, varied: bool = False):
    """Render line-item spending as labelled bars with separately known cents."""
    gold = invoice_record(invoice)
    values = [row["quantity"] * row["unit_minor"] for row in gold["line_items"]]
    image = Image.new("RGB", (896, 512), "#ffffff")
    draw = ImageDraw.Draw(image)
    accent = "#266978" if varied else "#27598b"
    order = list(reversed(range(len(values)))) if varied else list(range(len(values)))
    draw.text((48, 39), f"Line item spending · {invoice.currency}", fill=accent, font=_font(28, bold=True))
    draw.text((48, 80), "Total before tax per line item", fill="#53575b", font=_font(17))
    largest = max(values)
    rows = []
    for position, index in enumerate(order):
        row = gold["line_items"][index]
        value = values[index]
        y = 141 + 86 * position
        draw.text((51, y), row["description"], font=_font(19), fill="#222222")
        money = f"{value / 100:,.2f}"
        draw.text((734, y), money, font=_font(19, bold=True), fill="#222222")
        draw.rounded_rectangle((51, y + 34, 51 + int(value / largest * 650), y + 55), 4, fill=accent)
        rows.append({"description": row["description"], "line_total_minor": value})
    return image, {"currency": invoice.currency, "rows": rows}

## Which tokens receive a loss?

Both renders use the same image tokens. The prompt prefix must match the full rendered sequence exactly before the mask removes it.

In [4]:
"""Bounded Qwen2-VL training and evaluation shared by the document notebooks."""

from dataclasses import dataclass
from pathlib import Path
import random
from time import perf_counter

import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"
VISION_REV = "1860c46f12b7449d0f5b426a7474ab715342bc65"
VISION_SEED = 42
VISION_MAX_PIXELS = 576 * 28 * 28
VISION_MAX_OUTPUT = 220


@dataclass
class DocumentExample:
    key: str
    image: object
    instruction: str
    answer: str
    group: str


def processor_for_vision():
    return AutoProcessor.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False,
        min_pixels=4 * 28 * 28, max_pixels=VISION_MAX_PIXELS,
    )


def load_vision_base(trainable: bool = False):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime. This vision lab uses CUDA.")
    quantisation = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        VISION_MODEL, revision=VISION_REV, trust_remote_code=False, use_safetensors=True,
        torch_dtype=torch.float16, device_map={"": 0}, quantization_config=quantisation,
        attn_implementation="sdpa",
    )
    if trainable:
        return prepare_model_for_kbit_training(base, gradient_checkpointing_kwargs={"use_reentrant": False})
    return base.eval()


def adapter_for_vision(base):
    adapter = get_peft_model(
        base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                         lora_dropout=0.0, task_type="CAUSAL_LM"),
    )
    trainable = [name for name, value in adapter.named_parameters() if value.requires_grad]
    if not trainable or any(".visual." in name or "lora" not in name for name in trainable):
        raise RuntimeError("Expected language-backbone LoRA parameters only.")
    return adapter


def encode_image(processor, item: DocumentExample, assistant: str | None = None):
    chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": item.instruction}]}]
    if assistant is not None:
        chat.append({"role": "assistant", "content": [{"type": "text", "text": assistant}]})
    template = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=assistant is None)
    # Use the exact same pixels and processor settings in the prompt and full sequence.
    inputs = processor(text=[template], images=[item.image], padding=False, return_tensors="pt")
    return inputs.to("cuda")


def sft_feature(processor, item: DocumentExample):
    prompt = encode_image(processor, item)
    full = encode_image(processor, item, item.answer)
    labels = full["input_ids"].clone()
    prefix = prompt["input_ids"]
    width = prefix.shape[1]
    if width >= labels.shape[1] or not torch.equal(labels[:, :width], prefix):
        raise ValueError(f"Chat template changed its expanded image prefix for {item.key}.")
    labels[:, :width] = -100
    if not torch.any(labels[:, width:] != -100):
        raise ValueError("The prompt mask removed the entire target.")
    return full, labels


def fit_vision(adapter, processor, items, *, updates: int, accumulation: int = 2):
    random.seed(VISION_SEED)
    order = list(items)
    random.shuffle(order)
    if updates <= 0 or not order:
        raise ValueError("Vision training needs a positive update budget and at least one example.")
    adapter.train()
    optimiser = torch.optim.AdamW((p for p in adapter.parameters() if p.requires_grad), lr=1e-4)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = perf_counter()
    history = []
    for update in range(updates):
        optimiser.zero_grad(set_to_none=True)
        step_loss = 0.0
        for j in range(accumulation):
            features, labels = sft_feature(processor, order[(update * accumulation + j) % len(order)])
            scalar = adapter(**features, labels=labels, use_cache=False).loss
            if not torch.isfinite(scalar):
                raise RuntimeError(f"Non-finite vision SFT loss at step {update}.")
            (scalar / accumulation).backward()
            step_loss += scalar.detach().item() / accumulation
        norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        if not torch.isfinite(norm):
            raise RuntimeError("Non-finite adapter gradient.")
        optimiser.step()
        history.append(step_loss)
    torch.cuda.synchronize()
    adapter.eval()
    return {"updates": updates, "examples_seen": updates * accumulation, "seconds": perf_counter() - started,
            "peak_gib": torch.cuda.max_memory_allocated() / 2**30, "loss": history}


@torch.inference_mode()
def generate_vision(model, processor, example, *, max_new_tokens=VISION_MAX_OUTPUT, **options):
    inputs = encode_image(processor, example)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, **options)
    ids = output[0, inputs["input_ids"].shape[1]:]
    value = processor.tokenizer.decode(ids, skip_special_tokens=True).strip()
    stops = model.generation_config.eos_token_id
    eos = stops if isinstance(stops, (list, tuple)) else [stops]
    return {"text": value, "tokens": ids.cpu().tolist(), "complete": bool(len(ids) and ids[-1].item() in eos)}


def save_vision_adapter(model, path: str) -> Path:
    target = Path(path)
    model.save_pretrained(target, safe_serialization=True)
    if not (target / "adapter_model.safetensors").is_file():
        raise RuntimeError(f"Missing adapter weights at {target}")
    return target

In [5]:
import json
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
import hashlib
import random
import shutil

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook needs a Colab T4 runtime.")
torch.manual_seed(VISION_SEED)
START = perf_counter()
print("GPU", torch.cuda.get_device_name(0))
for package in ("torch", "transformers", "peft", "bitsandbytes", "Pillow"):
    print(package, version(package))

def paired_family_exact(before, after, get_exact, seed=42):
    baseline = {row["key"]: row for row in before}
    assert len(baseline) == len(before) == len(after)
    assert set(baseline) == {row["key"] for row in after}
    deltas = {}
    for current in after:
        previous = baseline[current["key"]]
        assert previous["family"] == current["family"]
        deltas.setdefault(current["family"], []).append(get_exact(current) - get_exact(previous))
    families = sorted(deltas)
    if len(families) < 2:
        return {"families":len(families),"interval_pp":None}
    rng = random.Random(seed)
    draws = []
    for _ in range(2000):
        rows = [delta for family in rng.choices(families, k=len(families)) for delta in deltas[family]]
        draws.append(100 * sum(rows) / len(rows))
    draws.sort()
    return {"families":len(families),
            "delta_pp":100 * sum(sum(values) for values in deltas.values()) / len(after),
            "interval_pp":[draws[49],draws[1949]]}

def feature_manifest(items, updates, accumulation):
    return {
        "model":VISION_MODEL, "revision":VISION_REV, "catalogue":CATALOG_VERSION,
        "updates":updates, "accumulation":accumulation, "seed":VISION_SEED,
        "features_sha256":hashlib.sha256(json.dumps([
            [item.key, item.group, item.instruction, item.answer,
             hashlib.sha256(item.image.tobytes()).hexdigest()] for item in items
        ],ensure_ascii=False).encode()).hexdigest(),
    }

def write_results(filename, payload):
    model = {"id": VISION_MODEL, "revision": VISION_REV}
    versions = {p: version(p) for p in ("torch","transformers","peft","bitsandbytes","Pillow","datasets")}
    experiment = {"elapsed_seconds": round(perf_counter()-START, 3),
                  "gpu": torch.cuda.get_device_name(0), "model": model,
                  "packages": versions, "catalogue": CATALOG_VERSION, "seed": VISION_SEED}
    with open(filename, "w", encoding="utf-8") as handle:
        json.dump({**experiment, **payload}, handle, indent=2, ensure_ascii=False)
    print(json.dumps({**experiment, **payload}, ensure_ascii=False))

GPU Tesla T4
torch 2.14.1+cu130
transformers 4.51.3
peft 0.15.2
bitsandbytes 0.50.2
Pillow 11.3.0


## What counts as a correct extraction?

Gold invoices use integer minor units and preserve row order. Chart answers use a separate schema and are scored as a transfer task. The same invoice lineage stays together across renders.

In [6]:
CHART_PROMPT = (
    "Read the bar chart. Return exactly one JSON object with currency and rows. "
    "Each row must contain description and line_total_minor in cents. Preserve visual row order."
)
def invoice_item(row):
    return DocumentExample(row.key,invoice_image(row),VISIBLE_CONTRACT,
                           json.dumps(invoice_record(row),ensure_ascii=False),row.family)

def chart_item(row, varied=False):
    image, answer=spend_chart(row,varied=varied)
    return DocumentExample(row.key,image,CHART_PROMPT,json.dumps(answer,ensure_ascii=False),row.family)

def chart_score(text, target):
    gold=json.loads(target)
    try:
        value=json.loads(text,object_pairs_hook=_strict_dict)
    except (TypeError,ValueError):
        return {"valid":False,"exact":False,"amount_error_minor":None}
    if (not isinstance(value,dict) or set(value)!={"currency","rows"}
            or type(value["currency"]) is not str or not isinstance(value["rows"],list)):
        return {"valid":False,"exact":False,"amount_error_minor":None}
    if any(not isinstance(row,dict) or set(row)!={"description","line_total_minor"} or
           type(row["line_total_minor"]) is not int or type(row["description"]) is not str for row in value["rows"]):
        return {"valid":False,"exact":False,"amount_error_minor":None}
    descriptions = [row["description"] for row in value["rows"]]
    if len(descriptions) != len(set(descriptions)):
        return {"valid":False,"exact":False,"amount_error_minor":None}
    by_name={row["description"]:row["line_total_minor"] for row in value["rows"]}
    errors=[abs(by_name[row["description"]]-row["line_total_minor"])
            for row in gold["rows"] if row["description"] in by_name]
    return {"valid":True,"exact":value==gold,"matched_labels":len(errors),"amount_error_minor":sum(errors)/len(errors) if errors else None}

def invoice_metrics(text, invoice):
    metrics = score_record(text, invoice)
    observed = parse_record(text)
    expected = invoice_record(invoice)["line_items"]
    predicted = observed["line_items"] if observed else []
    columns = ("description","quantity","unit_minor")
    matches = sum(gold[name] == guess[name]
                  for gold, guess in zip(expected, predicted) for name in columns)
    metrics["cell_f1"] = 2 * matches / (len(columns) * (len(expected) + len(predicted)))
    return metrics

train=[invoice_item(row) for row in examples_for("train",72)]+[chart_item(row) for row in examples_for("train",24)]
locked={split:examples_for(split,24) for split in ("familiar","novel","independent")}
assert {item.group for item in train}.isdisjoint(row.family for rows in locked.values() for row in rows)
# Reserve one printed arithmetic discrepancy in each unfamiliar slice.
for split in ("novel", "independent"):
    probe = next(row for row in examples_for(split) if row.printed_total_minor is not None)
    slice_rows = locked[split]
    if probe not in slice_rows:
        slot = next(i for i, row in enumerate(slice_rows) if row.family == probe.family)
        slice_rows[slot] = probe
chart_test=[chart_item(row,varied=True) for row in locked["novel"]]
print("Training examples, including one auxiliary chart slice:",len(train))
locked_keys = [row.key for rows in locked.values() for row in rows]
assert len(locked_keys) == len(set(locked_keys)) == 72
assert {item.key for item in train}.isdisjoint(locked_keys)
assert len(chart_test) == len(locked["novel"])
print("Held-out invoice counts:", {name:len(rows) for name,rows in locked.items()})

Training examples, including one auxiliary chart slice: 96
Held-out invoice counts: {'familiar': 24, 'novel': 24, 'independent': 24}


## Can one worst-width update fit on the T4?

Screen the longest processed training sequence before running the measured comparisons. Discard its weights and reset the seed before the scored fit.

In [7]:
from gc import collect
from importlib.metadata import version

def profile_t4(processor, items):
    free, _ = torch.cuda.mem_get_info()
    if free < 12 * 2**30:
        raise RuntimeError("The profile needs at least 12 GiB free on the T4.")

    def feature_width(item):
        features, labels = sft_feature(processor, item)
        return labels.shape[1], features["pixel_values"].shape[0]

    probe = max(items, key=feature_width)
    widths = feature_width(probe)
    torch.manual_seed(VISION_SEED)
    torch.cuda.empty_cache()
    starting_allocation = torch.cuda.memory_allocated()
    probe_model = adapter_for_vision(load_vision_base(trainable=True))
    fit = fit_vision(probe_model, processor, [probe], updates=1, accumulation=2)
    reserved_gib = torch.cuda.max_memory_reserved() / 2**30
    del probe_model
    collect()
    torch.cuda.empty_cache()
    retained_bytes = max(0, torch.cuda.memory_allocated() - starting_allocation)
    torch.cuda.reset_peak_memory_stats()
    torch.manual_seed(VISION_SEED)
    result = {
        "probe_key": probe.key, "sequence_tokens": widths[0], "image_patches": widths[1],
        "entry_free_gib": free / 2**30, "peak_reserved_gib": reserved_gib,
        "retained_bytes": retained_bytes, **fit,
    }
    Path("08_t4_profile.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    if (fit["seconds"] > 45 or fit["peak_gib"] > 10 or reserved_gib > 12
            or free - reserved_gib * 2**30 < 2 * 2**30 or retained_bytes > 128 * 2**20):
        raise RuntimeError("The profile exceeded its T4 time or memory budget.")
    return result

pins = {"transformers": "4.51.3", "peft": "0.15.2", "accelerate": "1.6.0",
        "bitsandbytes": "0.50.2", "pillow": "11.3.0"}
for name, expected in pins.items():
    if version(name) != expected:
        raise RuntimeError(f"Expected {name}=={expected}, got {version(name)}")
get_ipython().run_line_magic("config", "InlineBackend.figure_formats = ['svg']")
processor=processor_for_vision()
t4_profile=profile_t4(processor,train)

{
  "probe_key": "SF-652885",
  "sequence_tokens": 822,
  "image_patches": 2240,
  "entry_free_gib": 14.46075439453125,
  "peak_reserved_gib": 4.630859375,
  "retained_bytes": 17040384,
  "updates": 1,
  "examples_seen": 2,
  "seconds": 9.059662245999789,
  "peak_gib": 3.4266724586486816,
  "loss": [
    1.360404372215271
  ]
}


## How strong are the two baselines?

A rules baseline receives the verified visible text, an upper-bound input rather than noisy OCR. The frozen model receives pixels. Keep unmatched rules results in the denominator.

In [8]:
def model_scores(model):
    rows=[]
    for split, invoices in locked.items():
        for invoice in invoices:
            output=generate_vision(model,processor,invoice_item(invoice))
            rows.append({"split":split,"key":invoice.key,"family":invoice.family,
                         "metrics":invoice_metrics(output["text"],invoice),"complete":output["complete"],
                         "prediction":output["text"],"gold":invoice_record(invoice)})
    chart_rows=[]
    for item in chart_test:
        output=generate_vision(model,processor,item)
        chart_rows.append({"key":item.key,"metrics":chart_score(output["text"],item.answer),
                           "complete":output["complete"],"prediction":output["text"],"gold":json.loads(item.answer),"family":item.group})
    return {"invoices":rows,"charts":chart_rows}

def summarise(rows):
    result={}
    for split in locked:
        selected=[row for row in rows if row["split"]==split]
        result[split]={metric:sum(row["metrics"][metric] for row in selected)/len(selected)
                       for metric in ("exact","valid","fields","row_f1","cell_f1")}
        result[split]["count"]=len(selected)
    return result

rules_rows=[{"split":split,"key":row.key,"metrics":invoice_metrics(rules_extract(invoice_text(row)),row)}
            for split,invoices in locked.items() for row in invoices]
base=load_vision_base()
baseline=model_scores(base)
print("Rules with oracle text:",summarise(rules_rows))
print("Frozen VLM:",summarise(baseline["invoices"]))

Rules with oracle text: {'familiar': {'exact': 0.25, 'valid': 0.25, 'fields': 0.25, 'row_f1': 0.25, 'cell_f1': 0.25, 'count': 24}, 'novel': {'exact': 0.0, 'valid': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'cell_f1': 0.0, 'count': 24}, 'independent': {'exact': 0.0, 'valid': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'cell_f1': 0.0, 'count': 24}}
Frozen VLM: {'familiar': {'exact': 0.0, 'valid': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'cell_f1': 0.0, 'count': 24}, 'novel': {'exact': 0.0, 'valid': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'cell_f1': 0.0, 'count': 24}, 'independent': {'exact': 0.0, 'valid': 0.0, 'fields': 0.0, 'row_f1': 0.0, 'cell_f1': 0.0, 'count': 24}}


## What changed after supervised training?

Train a language-backbone adapter on the invoice and chart batches. Keep the prompt, processor, decoding and evaluation cases fixed.

In [9]:
from gc import collect
del base
collect()
torch.cuda.empty_cache()
model=adapter_for_vision(load_vision_base(trainable=True))
training=fit_vision(model,processor,train,updates=32,accumulation=2)
adapted=model_scores(model)
save_dir=save_vision_adapter(model,"08_structure_adapter")
adapter_sha = hashlib.sha256((save_dir/"adapter_model.safetensors").read_bytes()).hexdigest()
anchor=invoice_item(examples_for("familiar",1)[0])
handoff={"features":feature_manifest(train,32,2),
         "adapter_sha256":adapter_sha,"parity_key":anchor.key,
         "parity_tokens":generate_vision(model,processor,anchor)["tokens"]}
Path("08_handoff.json").write_text(json.dumps(handoff,indent=2),encoding="utf-8")
print("Adapter bytes SHA256:",adapter_sha)
print("Adapted:",summarise(adapted["invoices"]))
assert (save_dir/"adapter_config.json").is_file()

Adapter bytes SHA256: 592bbf3309b83409b446f8a0d5bcadff086a6d77eabd3f08a7b23c47565ad71c
Adapted: {'familiar': {'exact': 0.0, 'valid': 0.9583333333333334, 'fields': 0.041666666666666664, 'row_f1': 0.0, 'cell_f1': 0.013888888888888888, 'count': 24}, 'novel': {'exact': 0.0, 'valid': 1.0, 'fields': 0.05357142857142857, 'row_f1': 0.0, 'cell_f1': 0.020833333333333332, 'count': 24}, 'independent': {'exact': 0.0, 'valid': 0.9583333333333334, 'fields': 0.047619047619047616, 'row_f1': 0.0, 'cell_f1': 0.031018518518518518, 'count': 24}}


## Which gains survive a layout change?

Compare named slices and audit changed records. Chart accuracy and amount error answer different questions. Unparseable charts remain part of the exact-score denominator.

In [10]:
def exact_rows(result, split):
    return [row for row in result["invoices"] if row["split"] == split]

paired={split: paired_family_exact(exact_rows(baseline,split),exact_rows(adapted,split),
                                  lambda row: row["metrics"]["exact"])
        for split in locked}
print("Exact invoice changes in percentage points:",paired)
print("Charts, base and adapted:",
      [sum(r["metrics"]["exact"] for r in result["charts"])/len(result["charts"]) for result in (baseline,adapted)])
chart_exact_paired=paired_family_exact(baseline["charts"],adapted["charts"],
                                      lambda row: row["metrics"]["exact"])
write_results("08_structure_results.json",{"rules":rules_rows,"base":baseline,"adapted":adapted,
    "paired_exact":paired,"paired_chart_exact":chart_exact_paired,"training":training,"t4_profile":t4_profile,"adapter_sha256":adapter_sha,
    "locked_ids":{name:[x.key for x in rows] for name,rows in locked.items()}})

Exact invoice changes in percentage points: {'familiar': {'families': 8, 'delta_pp': 0.0, 'interval_pp': [0.0, 0.0]}, 'novel': {'families': 8, 'delta_pp': 0.0, 'interval_pp': [0.0, 0.0]}, 'independent': {'families': 4, 'delta_pp': 0.0, 'interval_pp': [0.0, 0.0]}}
Charts, base and adapted: [0.0, 0.0]
{"elapsed_seconds": 2129.522, "gpu": "Tesla T4", "model": {"id": "Qwen/Qwen2-VL-2B-Instruct", "revision": "1860c46f12b7449d0f5b426a7474ab715342bc65"}, "packages": {"torch": "2.14.1+cu130", "transformers": "4.51.3", "peft": "0.15.2", "bitsandbytes": "0.50.2", "Pillow": "11.3.0", "datasets": "3.6.0"}, "catalogue": "course-invoice-v1", "seed": 42, "rules": [{"split": "familiar", "key": "SF-228794", "metrics": {"valid": true, "exact": true, "fields": 1.0, "field_correct": {"invoice_id": true, "issued_on": true, "supplier": true, "currency": true, "subtotal_minor": true, "tax_minor": true, "total_minor": true}, "row_f1": 1.0, "cell_f1": 1.0}}, {"split": "familiar", "key": "SF-429916", "metrics":

## Run it yourself

Increase chart training exposure at a fixed update budget, vary the number of invoice rows, and compare both changes on the same layout hold-out. Keep the separate capstone release set closed.

Sources: [Qwen2-VL](https://huggingface.co/docs/transformers/v4.51.3/model_doc/qwen2_vl), [PEFT quantisation](https://huggingface.co/docs/peft/v0.15.0/developer_guides/quantization), [Nemotron document extraction](https://huggingface.co/blog/nvidia/building-on-nemotron-35-parse-open-data-fine-tunin).

[Read the result and compare the three systems](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/08-extract-tables-and-charts).